# TD3 — Climate Projections: Comparing Warming Scenarios

**Context:** Climate models produce projections for different emission scenarios, called **RCPs** (Representative Concentration Pathways). In this TD we will work with three:

| Scenario | Description |
|----------|-------------|
| **RCP 2.6** | Strong mitigation — warming below +2°C by 2100 |
| **RCP 4.5** | Moderate mitigation — warming around +2-3°C |
| **RCP 8.5** | Business as usual — warming above +4°C by 2100 |

We also have a **historical** dataset (1951-2005) to use as a reference.

The data comes from **DRIAS-2020**, a French climate service. It uses the model **CNRM-ALADIN63** driven by **CNRM-CM5**, bias-corrected with the **ADAMONT-SAFRAN** method, for a single grid point near **Paris** (48.9 N, 2.4 E).

Variables available:
- `tasmin` - daily minimum temperature (in Kelvin in the raw file)
- `tasmax` - daily maximum temperature (in Kelvin)
- `tas`    - daily mean temperature (in Kelvin)
- `prtot`  - total precipitation (in kg/m2/s)
- `prsn`   - solid precipitation (in kg/m2/s)


---
## Part 1 - Loading the Data

The data is stored as CSV files in the `data/` folder. Each file corresponds
to one scenario and contains daily climate data for a point near Paris.

We load them with `pd.read_csv()`, which we already know from TD2:

```python
df = pd.read_csv("data/my_file.csv")
```

The columns are already named: `date`, `lat`, `lon`, `tasmin`, `tasmax`, `tas`, `prtot`, `prsn`.


### 1.1 - Import the necessary libraries

In [5]:
...


Ellipsis

### 1.2 - Load the historical dataset

The data is stored in zip archives in the `data/` folder. The historical file contains `historical` in its name.

To read a CSV inside a zip without extracting it first, combine `glob`, `zipfile` and `pd.read_csv()`:

```python
import glob, zipfile

zip_path = glob.glob("data/*historical*.zip")[0]   # find the file
with zipfile.ZipFile(zip_path) as z:
    csv_name = z.namelist()[0]                     # get the CSV name inside the zip
    df = pd.read_csv(z.open(csv_name))
```

Explore the result with `.shape` and `.head()`.

In [ ]:
...

### 1.3 - Load the three projection datasets

Do the same for the three RCP scenarios. The zip file names contain `rcp26`, `rcp45`, or `rcp85`.

Store the results in `df_rcp26`, `df_rcp45`, `df_rcp85`.

In [ ]:
...

---
## Part 2 - Cleaning and Converting Units

### 2.1 - Parse dates

The `date` column contains integers like `19510101`.
Convert it to a proper datetime using:

```python
df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")
```

Apply this to **all four dataframes**. You can use a `for` loop over `[df_hist, df_rcp26, df_rcp45, df_rcp85]`.

In [8]:
...


Ellipsis

### 2.2 - Convert temperature from Kelvin to Celsius

The raw data is in **Kelvin** (K). Subtract 273.15 to get **Celsius** (degC).

Create three new columns `tasmin_c`, `tasmax_c`, `tas_c` for each of the four dataframes.

> What is the historical average temperature in degC? Is it consistent with what you know about Paris?

In [9]:
...

Ellipsis

### 2.3 - Convert precipitation from kg/m2/s to mm/day

Multiply by **86 400** (seconds per day) to get mm/day.

Create a column `precip_mm` for all four dataframes.


In [ ]:
for df in [df_hist, df_rcp26, df_rcp45, df_rcp85]:
    df["precip_mm"] = ...

print("Historical mean daily precipitation:", df_hist["precip_mm"].mean().round(2), "mm/day")

---
## Part 3 - Annual Temperature Trends

### 3.1 - Add a year column

Extract the year from the `date` column with `.dt.year` and store it in a new column `year`. Apply this to all four dataframes.

In [10]:
...


Ellipsis

### 3.2 - Compute annual mean temperature per scenario

Use `groupby("year")["tas_c"].mean().reset_index()` for each dataset and store the results (e.g. `annual_hist`, `annual_rcp26`, ...).

Inspect the result with `.tail()` to check the most recent years.

In [11]:
...

Ellipsis

### 3.3 - Plot all scenarios on a single chart

Call `plt.plot()` once per scenario before `plt.show()` to overlay them. Use different colours and add `plt.legend()`. Add labels with `plt.xlabel` and `plt.ylabel` 
> **Observation:** When do the scenarios start to diverge? What happens after 2050?

In [13]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.show()

<Figure size 1200x500 with 0 Axes>

### 3.4 - Annual precipitation per scenario

Compute the annual mean daily precipitation for each dataset with `groupby("year")["precip_mm"].mean()`.

Then plot all four scenarios on a single chart, using the same colour scheme as above (black / blue / orange / red).

> **Observation:** Is the precipitation trend as clear as the temperature trend? Why might that be?

In [ ]:
annual_pr_hist  = ...
annual_pr_rcp26 = ...
annual_pr_rcp45 = ...
annual_pr_rcp85 = ...

plt.figure(figsize=(12, 5))
...
plt.xlabel("Year")
plt.ylabel("Precipitation (mm/day)")
plt.title("Annual mean precipitation near Paris -- CNRM-CM5 / ALADIN63")
plt.legend()
plt.show()

### 3.5 - Smoothed precipitation (rolling average)

The annual series is noisy. Apply a **10-year centred rolling average** to each series to reduce noise:

```python
smoothed = annual_series.rolling(window=10, center=True).mean()
```

Replot the smoothed series. Do the scenarios diverge more clearly now?

In [ ]:
annual_pr_hist_s  = df_hist.groupby("year")["precip_mm"].mean().rolling(10, center=True).mean()
annual_pr_rcp26_s = ...
annual_pr_rcp45_s = ...
annual_pr_rcp85_s = ...

plt.figure(figsize=(12, 5))
plt.plot(annual_pr_hist_s.index,  annual_pr_hist_s.values,  color="black",  label="Historical")
plt.plot(...,..., color="blue",   label="RCP 2.6")
plt.plot(...,..., color="orange", label="RCP 4.5")
plt.plot(...,..., color="red",    label="RCP 8.5")
plt.xlabel("Year")
plt.ylabel("Precipitation (mm/day, 10-yr rolling)")
plt.title("Smoothed annual precipitation near Paris")
plt.legend()
plt.show()

### 3.6 - 20-year smoothed temperature: filtering natural variability

Annual temperature values jump year to year due to **natural climate variability** — random fluctuations driven by atmospheric dynamics, ocean circulation, volcanic eruptions, etc. This noise can mask the underlying long-term **anthropogenic signal**.

To reveal the forced trend, climate scientists apply a long rolling average (typically **20 years**, following IPCC climate normal conventions). Another classic approach is to average across many different climate models: since each model has its own random variability, it cancels out in the ensemble mean.

Apply a **20-year centred rolling average** to the annual mean temperature series and replot all scenarios.

> Does the divergence between RCP 8.5 and RCP 2.6 stand out more clearly now?

In [ ]:
annual_hist_smooth  = annual_hist.set_index("year")["tas_c"].rolling(20, center=True).mean()
annual_rcp26_smooth = ...
annual_rcp85_smooth = ...

plt.figure(figsize=(12, 5))
plt.plot(annual_hist_smooth.index,  annual_hist_smooth.values,  color="black",  label="Historical")
plt.plot(...,..., color="blue",   label="RCP 2.6")
plt.plot(..., ..., color="orange", label="RCP 4.5")
plt.plot(..., ..., color="red",    label="RCP 8.5")
plt.xlabel("Year")
plt.ylabel("20-yr smoothed temperature (degC)")
plt.title("Annual mean temperature (20-yr rolling average) near Paris")
plt.legend()
plt.show()

### 3.7 - Annual maximum temperature

Instead of the yearly mean, look at the **highest temperature recorded each year** (the annual maximum of `tasmax_c`). This captures how extreme the hottest days are becoming.

Use `groupby("year")["tasmax_c"].max()` for each dataset.

> How does the annual maximum evolve across scenarios compared with the mean?

In [ ]:
tmax_hist  = ...
tmax_rcp26 = ...
tmax_rcp45 = ...
tmax_rcp85 = ...

plt.figure(figsize=(12, 5))
...
plt.xlabel("Year")
plt.ylabel("Annual maximum temperature (degC)")
plt.title("Hottest day of the year near Paris")
plt.legend()
plt.show()

Again if you don't see clearly you can apply the smoothing

---
## Part 4 - Climate Indicators

Rather than annual means, climate scientists focus on **extreme events**.

### 4.1 - Hot days (tasmax > 30 degC)

A **hot day** is a day where the daily maximum temperature exceeds 30 degC.

Create a boolean column `hot_day` in `df_hist` and count how many hot days there were per year on average.


In [ ]:
df_hist["hot_day"] = ...

hot_days_annual = ...

print("Average hot days per year (historical):",...)

### 4.2 - Hot days in each scenario

Do the same for the three RCP scenarios. Compute the annual count of hot days for each, then plot all four series on a single chart.

**Python tip — iterating over a list with `for`**

Instead of writing the same line three times, you can loop over a list of dataframes:

```python
for df in [df_rcp26, df_rcp45, df_rcp85]:
    df["hot_day"] = df["tasmax_c"] > 30
```

Python reads this as: *"for each dataframe in this list, execute the indented block"*. The variable `df` takes the value of each element in turn. This keeps the code short and makes it easy to add a fourth scenario later without changing anything.

You can add `plt.axhline(hot_days_hist.mean(), color="grey", linestyle="--")` to mark the historical average as a horizontal reference.

In [ ]:


hot_days_rcp26 = ...
hot_days_rcp45 = ...
hot_days_rcp85 = ...

plt.figure(figsize=(12, 5))
...
plt.xlabel("Year")
plt.ylabel("Number of hot days (tasmax > 30 degC)")
plt.title("Hot days per year near Paris")
plt.legend()
plt.show()

### 4.3 - Warm nights (tasmin > 20 degC)

A **warm night** is a night where the minimum temperature stays above 20 degC. These are particularly uncomfortable in cities and can affect health.

Create a boolean column `warm_night` for each dataframe, count the warm nights per year, and plot all four series on a single chart.

In [ ]:

warm_hist  = ...
warm_rcp26 = ...
warm_rcp45 = ...
warm_rcp85 = ...

plt.figure(figsize=(12, 5))
...
plt.xlabel("Year")
plt.ylabel("Number of warm nights (tasmin > 20 degC)")
plt.title("Warm nights per year near Paris")
plt.legend()
plt.show()

### 4.4 - Snow days (supplementary)

The `prsn` variable gives **solid precipitation** (snow + sleet) in kg/m²/s.
Convert it to mm/day (× 86400), then define a **snow day** as any day with solid precipitation above **1 mm/day**.

Count the number of snow days per year for each scenario and plot the result.

> Does snow nearly disappear near Paris under RCP 8.5 by the end of the century?

In [ ]:
for df in [df_hist, df_rcp26, df_rcp45, df_rcp85]:
    df["prsn_mm"]  = df["prsn"] * 86400
    df["snow_day"] = ...

snow_hist  = df_hist.groupby("year")["snow_day"].sum()
snow_rcp26 = ...
snow_rcp45 = ...
snow_rcp85 = ...

plt.figure(figsize=(12, 5))
...
plt.xlabel("Year")
plt.ylabel("Number of snow days (prsn > 1 mm/day)")
plt.title("Snow days per year near Paris")
plt.legend()
plt.show()

### 4.5 - Smoothed snow days (rolling average)

The annual snow signal is also noisy. Apply a **10-year centred rolling average** to the snow day series to reveal the long-term trend, following the same approach as in 3.5.

In [ ]:
snow_hist_s  = snow_hist.rolling(10, center=True).mean()
snow_rcp26_s = ...
snow_rcp45_s = ...
snow_rcp85_s = ...

plt.figure(figsize=(12, 5))
...
plt.xlabel("Year")
plt.ylabel("Snow days (10-yr rolling)")
plt.title("Smoothed snow days per year near Paris")
plt.legend()
plt.show()

---
## Part 5 - Quantifying Change: Reference Period vs. Future

To measure how much the climate changes, scientists compare a **future period** to a **reference period**.

We will use:
- **Reference**: 1986-2005 (IPCC AR5 baseline period)
- **Near future**: 2021-2050 (projections)
- **Far future**: 2071-2100 (projections)

### 5.1 - Reference mean temperature (1986-2005)

Filter `df_hist` to keep only years 1986-2005, then compute the mean of `tas_c`.

In [ ]:
df_ref = df_hist[(df_hist["year"] >= 1986) & (df_hist["year"] <= 2005)]
ref_mean_temp = ...

print("Reference mean temperature (1986-2005):", round(ref_mean_temp, 2), "degC")

### 5.2 - Future mean temperatures

Compute the mean temperature for 2021-2050 and 2071-2100 for each scenario.

**Python tip — writing a helper function**

Without a function, you would repeat the same filtering and averaging three times per period — six times in total. A **function** lets you write the logic once and reuse it with different arguments:

```python
def period_mean(df, start, end):
    mask = (df["year"] >= start) & (df["year"] <= end)
    return df[mask]["tas_c"].mean()
```

- `def` declares the function and gives it a name (`period_mean`)
- `df`, `start`, `end` are **parameters** — placeholders for the actual values passed when you call the function
- `return` sends the result back to the caller

You can then call it like any built-in function:

```python
period_mean(df_rcp26, 2021, 2050)   # mean temperature of RCP 2.6 over 2021-2050
```

**Rule of thumb:** if you find yourself copy-pasting the same block of code more than twice, write a function instead.

In [ ]:
def period_mean(df, start, end):
    ...

near_future = {
    "RCP 2.6": period_mean(df_rcp26, 2021, 2050),
    "RCP 4.5": ...,
    "RCP 8.5": ...,
}

far_future = {
    "RCP 2.6": period_mean(df_rcp26, 2071, 2100),
    "RCP 4.5": ...,
    "RCP 8.5": ...,
}

print("Near future (2021-2050):", {k: round(v, 2) for k, v in near_future.items()})
print("Far future  (2071-2100):", {k: round(v, 2) for k, v in far_future.items()})

### 5.3 - Compute the warming (delta T)

Subtract the reference mean temperature from each future value to get the temperature **change**.


In [ ]:
delta_near = ...
delta_far  = ...

print("Warming 2021-2050:", {k: round(v, 2) for k, v in delta_near.items()})
print("Warming 2071-2100:", {k: round(v, 2) for k, v in delta_far.items()})

### 5.4 - Bar chart of future warming

Create a grouped bar chart showing the warming for both the near and far future, per scenario.

Use `numpy` to place bars side by side:

```python
import numpy as np
x = np.arange(len(scenarios))   # [0, 1, 2]
width = 0.35
plt.bar(x - width/2, list(delta_near.values()), width, label="2021-2050")
plt.bar(x + width/2, list(delta_far.values()),  width, label="2071-2100")
plt.xticks(x, scenarios)
```


In [ ]:
import numpy as np

scenarios = list(delta_near.keys())
x = np.arange(len(scenarios))
width = 0.35

colors_near = ["#6baed6", "#fdae6b", "#fc8d59"]   # muted blue / orange / red (near future)
colors_far  = ["#08519c", "#e6550d", "#b30000"]   # deep  blue / orange / red (far future)

plt.figure(figsize=(8, 5))
plt.bar(x - width/2, ..., width, label="2021-2050", color=colors_near, edgecolor="grey",  linewidth=0.7)
plt.bar(x + width/2, ..., width, label="2071-2100", color=colors_far,  edgecolor="black", linewidth=0.7)
plt.xticks(x, scenarios)
plt.ylabel("Temperature change (degC) relative to 1981-2010")
plt.title("Projected warming near Paris by scenario")
plt.legend()
plt.axhline(0, color="black", linewidth=0.8)
plt.show()

---
## Going Further — Explore DRIAS

This TD uses a **single climate model** (CNRM-CM5) at a **single location** (near Paris). Real climate assessments always rely on **ensembles** of many models: the spread across models captures the remaining uncertainty in climate projections, and averaging across them reduces the influence of each model's internal variability.

**DRIAS** (drias-les-futurs.fr) is the French national portal for regional climate projections. It gives access to:
- Several regional climate models (CNRM-ALADIN63, IPSL-WRF381P, HadREM3-GA7-05, ...)
- All RCP scenarios
- Any location in metropolitan France (point or region)

**Challenge:** Download data for a different city (e.g. **Marseille**, **Lyon**, or **Brest**) or a different model, and redo some of the analyses above.

Things to investigate:
- Does RCP 8.5 still show the strongest warming everywhere?
- Does a coastal city warm differently from an inland city?
- Do hot days increase by the same amount in southern France as in Paris?

---
## Congratulations!

You have completed TD3. You have learned how to:
- Load climate data from CSV files
- Convert physical units (K to degC, kg/m²/s to mm/day)
- Compare multiple emission scenarios on the same chart
- Filter natural variability with rolling averages
- Compute climate indicators (hot days, warm nights, snow days)
- Quantify future warming relative to a reference period

**Discussion questions:**
1. By how much does Paris warm under RCP 8.5 by the end of the century relative to 1981-2010? How does this compare to RCP 2.6?
2. How does the number of hot days evolve differently across scenarios? What about warm nights?
3. Why is it important to smooth with a rolling average (or average across many models) rather than looking at raw annual values?
4. What are the limits of working with a single climate model and a single grid point?